# Task B — train on train + labelled validation, predict released test

This notebook trains the current best Task B recipe on the original 3,159 labelled rows plus the 395 released labelled validation rows, then predicts the 395-row released test file. It uses full-data TAPT, one-layer reinitialization, R-Drop 0.5, and five seeds. The output ZIP targets the released test IDs, not the closed CodaBench validation inputs. Expected runtime is roughly 3–5 hours on a Kaggle T4 ×2.

In [ ]:
import os, pathlib, shutil, subprocess, sys, zipfile, json
WORK = "/kaggle/working/hastika"
if os.path.isdir(WORK + "/.git"):
    subprocess.run(["git", "-C", WORK, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "-q", "-b", "task-b", "--depth", "1", "https://github.com/robinpnalex/Hastika-ICON2026.git", WORK], check=True)
os.chdir(WORK)
os.environ["PYTHONPATH"] = os.path.join(WORK, "src")
sys.path.insert(0, os.path.join(WORK, "src"))
pathlib.Path("artifacts/logs").mkdir(parents=True, exist_ok=True)
subprocess.run('pip install -q emoji ftfy sentencepiece protobuf "transformers>=4.45,<6"', shell=True, check=True)
import pandas as pd, torch
assert torch.cuda.is_available(), "enable a GPU"
print("gpu:", torch.cuda.get_device_name(0))

def run(cmd, log=None):
    print("$", " ".join(map(str, cmd)), flush=True)
    fh = open(log, "w") if log else None
    p = subprocess.Popen([str(x) for x in cmd], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        sys.stdout.write(line)
        if fh: fh.write(line)
    p.wait()
    if fh: fh.close()
    if p.returncode: raise RuntimeError(f"exit {p.returncode}: {cmd}")


## 1. Combine labelled rows and stage the released test comments

In [ ]:
from pathlib import Path
train_path = Path("data/raw/multiclass_train.csv")
val_path = Path("data/raw/hastika_multiclass_validation.csv")
test_path = Path("data/raw/hastika_multiclass_test.csv")
combined_path = Path("data/derived/task_b_train_plus_validation.csv")
combined_path.parent.mkdir(parents=True, exist_ok=True)
combined = pd.concat([pd.read_csv(train_path), pd.read_csv(val_path)], ignore_index=True)
combined = combined.drop_duplicates("id", keep="first").reset_index(drop=True)
assert combined["id"].is_unique and len(combined) == 3554
combined.to_csv(combined_path, index=False)
print("combined labelled rows:", len(combined), "test rows:", len(pd.read_csv(test_path)))
CANON_TEST = Path("data/raw/multiclass_validation_inputs.csv")
shutil.copy2(CANON_TEST, CANON_TEST.with_suffix(".csv.before_final"))
shutil.copy2(test_path, CANON_TEST)


## 2. TAPT and five-seed full-data training

In [ ]:
TAPT_OUT = "artifacts/runs/tapt-taskb-trainval"
TAPT_LOG = "artifacts/logs/taskb_trainval_tapt.log"
run([sys.executable, "-u", "-m", "hastika.task_b.tapt", "--model", "google/muril-base-cased", "--corpus", str(combined_path), "data/external/offenseval_kn.csv", "--val-frac", "0", "--min-words", "1", "--no-dedupe", "--epochs", "8", "--out", TAPT_OUT], log=TAPT_LOG)
assert (Path(TAPT_OUT) / "config.json").exists()
TAG = "b_reinit1_rdrop_trainval_test"
RUN_DIR = Path("artifacts/runs") / TAG
run([sys.executable, "-u", "-m", "hastika.task_b.train", "--tag", TAG, "--model", TAPT_OUT, "--train-data", str(combined_path), "--folds", "1", "--no-dedupe", "--reinit-layers", "1", "--rdrop", "0.5", "--aux-weight", "0", "--seeds", "42", "43", "44", "45", "46", "--epochs", "6"], log=f"artifacts/logs/{TAG}.log")
assert (RUN_DIR / "test_probs.npy").exists() and (RUN_DIR / "predictions.csv").exists()
print("five-seed train+validation fit completed")


## 3. Restore the old input path and package for the released test file

In [ ]:
PRED = RUN_DIR / "predictions.csv"
ZIP = Path("/kaggle/working/task_b_trainval_test.zip")
shutil.move(str(CANON_TEST.with_suffix(".csv.before_final")), str(CANON_TEST))
run([sys.executable, "-m", "hastika.common.submission", "--task", "b", "--pred", str(PRED), "--inputs", str(test_path), "--out", str(ZIP)])
with zipfile.ZipFile(ZIP) as z: assert z.namelist() == ["predictions.csv"]
OUT = Path("/kaggle/working/task_b_trainval_test_outputs")
OUT.mkdir(parents=True, exist_ok=True)
shutil.copy2(ZIP, OUT / ZIP.name); shutil.copy2(PRED, OUT / PRED.name)
shutil.copy2(RUN_DIR / "test_probs.npy", OUT / "test_probs.npy")
json.dump({"train_rows": len(combined), "seeds": [42,43,44,45,46], "reinit_layers": 1, "rdrop": 0.5}, open(OUT / "config.json", "w"), indent=2)
print("READY TO DOWNLOAD:", ZIP)
